In [23]:
from enum import Enum
import time
import numpy as np
import pandas as pd
import tqdm as notebook_tqdm
import argparse
import csv
import datetime
import json
import os
import shutil
import utils
from constants import *
from run import *
from models import *
from sim import *

import torch
import gpytorch

from sklearn.metrics import mean_squared_error  # Importing mean_squared_error
from scipy.stats import multivariate_normal

In [24]:
#import AA file for use

csv_file_path = "/Users/juar705/Downloads/mock_data.csv"
df = pd.read_csv(csv_file_path)
truncated_df = df.head(501)
del truncated_df['environment']

In [25]:
'''seperate into X_train and y_train sets
    X_train will be the amino acid columns 
    y_train will be the growth column'''

AA_columns = AA_SHORT
growth_columns = 'growth'

X_train = truncated_df[AA_columns].to_numpy()
y_train = truncated_df[growth_columns].to_numpy()

In [26]:
def sample_GP(model, likelihood, X, n_samples=1):
    
    # Convert data to tensor
    train_x = torch.tensor(X_train, dtype=torch.float)
    train_y = torch.tensor(y_train, dtype=torch.float)
    
    test_x = torch.tensor(X, dtype=torch.float32)
    
    # Initialize likelihood and model method
    likelihood = gpytorch.likelihoods.GaussianLikelihood()
    model = gpr.ExactGPModel(train_x, train_y, likelihood)
    
    # Load the state dictionaries from previously saved files
    model.load_state_dict(torch.load('/Users/juar705/BacterAI_pnnl/mockrun/Round1/gpr_model/gpr_model.pth'))
    likelihood.load_state_dict(torch.load('/Users/juar705/BacterAI_pnnl/mockrun/Round1/gpr_model/gpr_likelihood.pth'))
    
    # Set model and likelihood to evaluation mode
    model.eval()
    likelihood.eval()

    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        observed_pred = likelihood(model(test_x))
     
    # Get the mean and covariance
    mean = observed_pred.mean.numpy()
    cov = observed_pred.covariance_matrix.numpy()
    
    # Sample from the multivariate normal distribution
    samples = np.atleast_1d(multivariate_normal.rvs(mean, cov, size=n_samples))
    variances = np.diag(cov)
    
    return samples, variances

In [27]:
class GPRModel(Model):
    def __init__(self, model_path):
        # self.activate_R()
        self.model = []
        self.likelihood = []
        self.model_path = model_path
        self.is_trained = False
        super().__init__(self, ModelType.GPR)
        
    @classmethod
    def load_trained_models(cls, models_path):
        obj = cls(models_path)

        for filename in os.listdir(models_path):
            if "model" in filename:
                model = torch.load(os.path.join(models_path, filename), map_location=torch.device(net.DEVICE))
                obj.model.append(model)
            if "likelihood" in filename:
                likelihood = torch.load(os.path.join(models_path, filename), map_location=torch.device(net.DEVICE))
                obj.likelihood.append(likelihood)

        obj.is_trained = True
        return obj
    
    def check_path(self):
        if not os.path.exists(self.model_path):
            os.makedirs(self.model_path)

    def train(self, X_train, y_train, **kwargs):
        # X_trainR = robjects.r.matrix(
        #     X_train, nrow=X_train.shape[0], ncol=X_train.shape[1]
        # )
        # y_trainR = robjects.r.matrix(y_train, nrow=y_train.shape[0], ncol=1)
        # self.model = self.gpr_lib.train_new_GP(X_trainR, y_trainR)
        self.check_path()
        self.model, self.likelihood = gpr.train_new_GP(X_train, y_train, self.model_path, **kwargs)
        self.is_trained = True

    def evaluate(self, X, clip=True, n=1):
        # X_evalR = robjects.r.matrix(X, nrow=X.shape[0], ncol=X.shape[1])
        if not self.is_trained:
            raise Exception("GPR model needs to be trained before evaluating.")
        
        #removed gpr to obtain straight from the notebook rather than the file
        samples, variances  = sample_GP(self.model, self.likelihood, X, n)
        # Do we want to clip samples?
        if clip:
            samples = np.clip(samples, 0, 1)
        return samples, variances

In [28]:
with open('config.json', 'r') as file:
    config = json.load(file)
MODEL_TYPE = ModelType(0)   
NEW_ROUND_N = 1
EXPT_FOLDER = config["experiment_path"]
new_round_folder = os.path.join(EXPT_FOLDER, f"Round{NEW_ROUND_N}")
if MODEL_TYPE == ModelType.GPR:
    models_folder = os.path.join(new_round_folder, f"gpr_model")
    model = GPRModel(models_folder)
    model.train(X_train, y_train)      

Iter 1/100 | Train loss: 1.2109
Iter 2/100 | Train loss: 1.1463
Iter 3/100 | Train loss: 1.0882
Iter 4/100 | Train loss: 1.0369
Iter 5/100 | Train loss: 0.9902
Iter 6/100 | Train loss: 0.9450
Iter 7/100 | Train loss: 0.8992
Iter 8/100 | Train loss: 0.8518
Iter 9/100 | Train loss: 0.8030
Iter 10/100 | Train loss: 0.7530
Iter 11/100 | Train loss: 0.7024
Iter 12/100 | Train loss: 0.6519
Iter 13/100 | Train loss: 0.6021
Iter 14/100 | Train loss: 0.5535
Iter 15/100 | Train loss: 0.5067
Iter 16/100 | Train loss: 0.4620
Iter 17/100 | Train loss: 0.4199
Iter 18/100 | Train loss: 0.3807
Iter 19/100 | Train loss: 0.3447
Iter 20/100 | Train loss: 0.3121
Iter 21/100 | Train loss: 0.2831
Iter 22/100 | Train loss: 0.2580
Iter 23/100 | Train loss: 0.2367
Iter 24/100 | Train loss: 0.2195
Iter 25/100 | Train loss: 0.2062
Iter 26/100 | Train loss: 0.1969
Iter 27/100 | Train loss: 0.1912
Iter 28/100 | Train loss: 0.1889
Iter 29/100 | Train loss: 0.1892
Iter 30/100 | Train loss: 0.1916
Iter 31/100 | Train

In [29]:
#Create an array of ones for down direction or 0 for up direction

n_ingredients = len(AA_SHORT)
batch_size = config["batch_size"]
DIRECTION = SimDirection(config["direction"])


def media_array(n_ingredients,direction):
    if DIRECTION == SimDirection.DOWN:
        media = np.ones(n_ingredients)
        direction = SimDirection.DOWN
    elif DIRECTION == SimDirection.UP:
        media = np.zeros(n_ingredients)
        direction = SimDirection.UP
    else:
        raise ValueError("Error") 
    return media

media = media_array(n_ingredients, DIRECTION)
print(f"Media Array:\n{media}")

Media Array:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [30]:
def make_batch(
    model,
    media,
    new_round_n,
    batch_size,
    sim_types,
    rollout_trajectories,
    threshold,
    timeout=60,
    unique=True,
    direction=SimDirection.DOWN,
    go_beyond_frontier=True,
    used_experiments=None,
    redo_experiments=None,
):
    """ Make a new BacterAI batch; the main function that calls the simulation loops. """
    sim_types=sim_types
    n_types = len(sim_types)
    n_exps = batch_size // n_types
    batch_set = used_experiments
    sub_batches = []
    all_metrics = {}
    for idx, sim_type in enumerate(sim_types):
        if idx == n_types - 1:
            n_exps = batch_size - sum([len(x) for x in sub_batches])
        print(idx, sim_type, batch_size, n_exps, sum([len(x) for x in sub_batches]))
        batch, batch_set, metrics = perform_simulations(
            model,
            media,
            n_exps,
            threshold,
            sim_type,
            direction,
            new_round_n,
            unique=unique,
            timeout=timeout,
            batch_set=batch_set,
            n_rollout_trajectories=rollout_trajectories,
            go_beyond_frontier=go_beyond_frontier,
        )
        sub_batches.append(batch)
        all_metrics[sim_type.name] = metrics

    batch = pd.concat([redo_experiments] + sub_batches, ignore_index=True)
    return batch, batch_set, all_metrics

In [31]:
trained_set = pd.DataFrame(np.hstack((X_train, y_train.reshape(-1,1))))
used_experiments = set(map(tuple,trained_set.to_numpy()))
batch_data= used_experiments

#set parameters needed for make batch function
sim_types=[SimType(2)]
rollout_trajectories=config["n_rollouts"]
threshold=config['grow_threshold']
timeout=60 * 3
unique=True 
direction = SimDirection(0)
go_beyond_frontier=config['beyond_frontier']

# Make batch the main function that calls to make all simulations from perform simulations function
batch, batch_set, all_metrics = make_batch(
    model=GPRModel.load_trained_models(models_folder),
    media=media,
    new_round_n=NEW_ROUND_N,
    batch_size=batch_size,
    sim_types=sim_types,
    rollout_trajectories=rollout_trajectories,
    threshold=threshold,
    timeout=timeout,
    unique=unique,
    direction=direction,
    go_beyond_frontier=go_beyond_frontier,
    used_experiments=None,
    redo_experiments=None,)

0 SimType.ROLLOUT 20 20 0



Performing ROLLOUT Sims (DOWN):   0%|                    | 0/20 [00:00<?, ?it/s]/Users/juar705/miniconda3/lib/python3.13/site-packages/scipy/stats/_multivariate.py:762: RuntimeWarning: covariance is not symmetric positive-semidefinite.
  out = random_state.multivariate_normal(mean, cov, size)

Performing ROLLOUT Sims (DOWN) (1 loops):   5%|  | 1/20 [00:01<00:25,  1.34s/it]


	ADDED: [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	ADDED: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND



Performing ROLLOUT Sims (DOWN) (2 loops):  15%|▎ | 3/20 [00:02<00:14,  1.19it/s]


	ADDED: [0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	ADDED: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND


...


	ADDED: [0 0 0 0 0 1 1 0 0 0 0 1 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND



Performing ROLLOUT Sims (DOWN) (22 loops):  90%|▉| 18/20 [00:28<00:02,  1.42s/it


	ADDED: [0 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 0 1] - FRONTIER

	ADDED: [0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 1] - BEYOND

	EXISTS: [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND



Performing ROLLOUT Sims (DOWN) (25 loops): 100%|█| 20/20 [00:32<00:00,  1.62s/it


	ADDED: [0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER
perform_simulations function took 32430.17 ms


In [32]:
print(f"Batch:\n,{batch}")

Batch:
,    0  1  2  3  4  5  6  7  8  9  ...  17  18  19     type  direction  \
0   0  0  0  0  0  0  1  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
1   0  0  0  0  0  0  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
2   0  0  0  0  0  1  0  0  1  0  ...   0   0   0  ROLLOUT       DOWN   
3   0  0  0  0  0  1  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
4   0  0  0  1  0  0  1  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
5   0  0  0  1  0  0  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
6   0  0  0  0  0  0  1  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
7   0  0  0  0  0  1  1  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
8   0  0  0  0  0  1  0  0  1  0  ...   0   0   1  ROLLOUT       DOWN   
9   0  0  0  0  0  0  0  0  1  0  ...   0   0   1  ROLLOUT       DOWN   
10  0  0  0  0  0  1  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
11  0  0  0  0  0  1  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
12  1  0  0  0  0  0  1  0  0  0  ...   0  

In [33]:
print(f"Batch Set:\n{batch_set}")

Batch Set:
{(np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)), (np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)), (np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)), (np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int

In [34]:
print(f"Metrics:\n{all_metrics}")

Metrics:
{'ROLLOUT': {'k_history': [], 'count_history': [], 'k_avg': 'n/a', 'count_avg': 'n/a', 'total_loops_count': 25, 'time_to_finish_sec': 32.43}}
